# ORB-BSCNN — Option B extension experiments

Extends `ORB_BSCNN_Reproducible.ipynb` with the experiments needed to turn the paper from "an architecture that does not beat the baselines" into a study with a clear contribution. Every original model, seed and fold is kept, so the original rows of Table 2 are reproduced exactly (same library versions permitting).

**New experiments**

1. **True simplicial propagation.** A node + edge model: the edge branch filters edge signals with the 1st-order Hodge Laplacian, split into its lower part `B1ᵀB1` and upper part `B2B2ᵀ` (Yang, Isufi and Leus, 2022). The upper part is the only place the triangles enter, so the ablation *without* it is the direct test of whether 2-simplices add anything.
2. **Graph neural network baselines** on the same feature graph: GCN (Kipf and Welling) and GAT (Veličković et al.), full precision, same readout and head.
3. **Graph-structure controls:** the feature model on a random graph with the same number of edges, and on an empty graph. If these match the correlation graph, the graph structure is not what helps.
4. **XGBoost baseline** (fixed, untuned settings, like the other baselines).
5. **Nested selection of τ** (inner 3-fold CV on each outer training fold), removing the model-selection limitation.
6. **A large dataset:** the Kaggle Cardiovascular Disease dataset (70,000 patients, 11 variables), which gives the statistical power that Cleveland (n = 297) lacks.

**How to run (Colab):** Runtime → Run all. For the large dataset you need a Kaggle API token: Kaggle → Settings → API → *Create New Token* downloads `kaggle.json`; upload it when the notebook asks. Set `QUICK_TEST = True` first to check that everything runs in a few minutes.

**Runtime:** Cleveland runs on CPU, about 30–60 min (nested τ is the slowest part). The large dataset needs a **GPU runtime** (T4), roughly 1–2 h with `LARGE_REPEATS = 2`. You can run the two parts in separate sessions with `RUN_CLEVELAND` / `RUN_LARGE`.

## 1. Setup and configuration

In [ ]:
import numpy as np, pandas as pd, networkx as nx
import scipy.sparse as sp
import os, random, time, platform, subprocess, shutil
import sklearn, scipy
from scipy import stats
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import kneighbors_graph
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
try:
    import xgboost
except ImportError:
    subprocess.run(["pip", "install", "-q", "xgboost"], check=True)
    import xgboost
from xgboost import XGBClassifier

QUICK_TEST     = os.environ.get("QUICK_TEST", "0") == "1"   # True: tiny run to check the pipeline end to end
RUN_CLEVELAND  = True
RUN_LARGE      = True
RUN_NESTED     = True    # nested selection of the correlation threshold (slowest part)

BASE_SEED      = 42
N_REPEATS      = 5       # Cleveland: repeats of 5-fold CV -> 25 paired folds (as in the manuscript)
LARGE_REPEATS  = 2       # large dataset: 2 x 5 = 10 paired folds (enough at n = 70,000)
N_SPLITS       = 5
NESTED_INNER   = 3
EPOCHS         = 150
CORR_THRESHOLD = 0.15
K_NEIGHBOURS   = 8
THRESHOLDS     = [0.05, 0.10, 0.15, 0.20, 0.30]
SPARSE_MIN_NODES = 2000  # patient graphs at least this large use a sparse Laplacian
CARDIO_CSV     = os.environ.get("CARDIO_CSV", "cardio_train.csv")
CARDIO_SUBSAMPLE = None  # e.g. 10000 for a faster large-data run
METRICS        = ["Accuracy", "Sensitivity", "Specificity", "F1", "AUC"]

if QUICK_TEST:
    N_REPEATS, LARGE_REPEATS, EPOCHS, CARDIO_SUBSAMPLE = 1, 1, 20, 3000
    THRESHOLDS = [0.10, 0.30]

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cpu")          # CPU on purpose: repeatable results
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

try:
    display
except NameError:              # running as a plain script
    display = print

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

versions = pd.DataFrame({"package": ["python", "torch", "scikit-learn", "xgboost", "numpy", "pandas", "networkx", "scipy"],
                         "version": [platform.python_version(), torch.__version__, sklearn.__version__, xgboost.__version__,
                                     np.__version__, pd.__version__, nx.__version__, scipy.__version__]})
versions

In [ ]:
FG, PG = "ORB-BSCNN (feature graph)", "ORB-BSCNN (patient graph)"
SC     = "ORB-BSCNN (simplicial: node + edge)"

# Models that share an id get the same seed, so their initial weights match and only the named component differs.
MODEL_IDS = {PG: 1, FG: 2, "MLP": 3, "Random Forest": 4, "Logistic Regression": 5,
             "Ablation: feature graph, no binarization": 2,
             "Ablation: feature graph, no residual": 2,
             "Ablation: patient graph, 2 blocks": 1,
             SC: 6,
             "Ablation: simplicial, no triangles (no L1 up)": 6,
             "Control: feature model, random graph": 2,
             "Control: feature model, empty graph": 2,
             "GCN (feature graph)": 7, "GAT (feature graph)": 8, "XGBoost": 9,
             "ORB-BSCNN (feature graph, nested τ)": 10}

def run_seed(repeat, fold, model_name):
    """One fixed seed per (repeat, fold, model)."""
    return BASE_SEED + 10000 * repeat + 100 * fold + MODEL_IDS[model_name]

## 2. Data

**Cleveland** is loaded exactly as in the original notebook. **Cardiovascular Disease** (Kaggle, `sulianova/cardiovascular-disease-dataset`): age is converted from days to years, and records with implausible measurements are removed with pre-specified rules (systolic 80–250 mmHg, diastolic 40–160 mmHg, systolic > diastolic, height 120–220 cm, weight 30–200 kg).

In [ ]:
COLS = ["age","sex","cp","trestbps","chol","fbs","restecg","thalach",
        "exang","oldpeak","slope","ca","thal","target"]
FEATURE_COLS = COLS[:-1]
BASE_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/"

df = pd.read_csv(BASE_URL + "processed.cleveland.data", names=COLS, na_values="?").dropna().reset_index(drop=True)
df["target"] = (df["target"] > 0).astype(int)
X = df[FEATURE_COLS].values.astype(np.float32)
y = df["target"].values.astype(np.int64)
print("Cleveland complete cases:", X.shape, " class counts:", np.bincount(y))

In [ ]:
CARDIO_SLUG = "sulianova/cardiovascular-disease-dataset"
CARDIO_FEATURES = ["age", "gender", "height", "weight", "ap_hi", "ap_lo", "cholesterol", "gluc", "smoke", "alco", "active"]

def fetch_cardio(path=CARDIO_CSV):
    if os.path.exists(path):
        return path
    token = os.path.expanduser("~/.kaggle/kaggle.json")
    if not os.path.exists(token):
        from google.colab import files          # outside Colab: put kaggle.json in ~/.kaggle/ or cardio_train.csv here
        print("Upload kaggle.json (Kaggle -> Settings -> API -> Create New Token)")
        files.upload()
        os.makedirs(os.path.dirname(token), exist_ok=True)
        shutil.move("kaggle.json", token)
        os.chmod(token, 0o600)
    subprocess.run(["kaggle", "datasets", "download", "-d", CARDIO_SLUG, "--unzip", "-p", "."], check=True)
    return path

def load_cardio(path):
    d = pd.read_csv(path, sep=";")
    missing = set(CARDIO_FEATURES + ["cardio"]) - set(d.columns)
    assert not missing, f"unexpected columns in {path}: missing {missing}"
    n0 = len(d)
    d["age"] = d["age"] / 365.25
    keep = (d.ap_hi.between(80, 250) & d.ap_lo.between(40, 160) & (d.ap_hi > d.ap_lo)
            & d.height.between(120, 220) & d.weight.between(30, 200))
    d = d[keep].reset_index(drop=True)
    print(f"Cardio: {n0} records, {n0 - len(d)} removed by the plausibility rules, {len(d)} kept")
    if CARDIO_SUBSAMPLE and CARDIO_SUBSAMPLE < len(d):
        d = d.sample(CARDIO_SUBSAMPLE, random_state=BASE_SEED).reset_index(drop=True)
        print(f"Subsampled to {len(d)} records")
    return d[CARDIO_FEATURES].values.astype(np.float32), d["cardio"].values.astype(np.int64)

if RUN_LARGE:
    X_cardio, y_cardio = load_cardio(fetch_cardio())
    print("Cardio:", X_cardio.shape, " class counts:", np.bincount(y_cardio))

## 3. Complex construction

Unchanged from the original notebook, plus: the lower and upper 1st-order Hodge Laplacians for the edge branch (each scaled by its largest eigenvalue), the GCN adjacency, controls built from a random or empty graph, and a sparse Laplacian for large patient graphs (identical in value to the dense one, `B1B1ᵀ = D − A`).

In [ ]:
def _complex_from_graph(G, n_nodes, with_B2=False):
    edges = [tuple(sorted(e)) for e in G.edges()]
    B1 = np.zeros((n_nodes, len(edges)), dtype=np.float32)
    for j, (u, v) in enumerate(edges):
        B1[u, j] = -1.0; B1[v, j] = 1.0
    triangles = []
    for c in nx.enumerate_all_cliques(G):        # yields cliques in order of size
        if len(c) > 3: break
        if len(c) == 3: triangles.append(tuple(sorted(c)))
    cplx = {"n_nodes": n_nodes, "n_edges": len(edges), "n_tri": len(triangles), "B1": B1, "edges": edges}
    if with_B2:
        edge_index = {e: i for i, e in enumerate(edges)}
        B2 = np.zeros((len(edges), len(triangles)), dtype=np.float32)
        for t, (a, b, c) in enumerate(triangles):
            B2[edge_index[(a, b)], t] = 1.0; B2[edge_index[(b, c)], t] = 1.0; B2[edge_index[(a, c)], t] = -1.0
        cplx["B2"] = B2
    return cplx

def build_patient_complex(Xd, k=K_NEIGHBOURS, with_B2=False):
    """Nodes = patients, edges = k nearest neighbours in standardized feature space."""
    knn = kneighbors_graph(Xd, n_neighbors=k, mode="connectivity", include_self=False)
    G = nx.Graph(nx.from_scipy_sparse_array(knn))
    return _complex_from_graph(G, Xd.shape[0], with_B2)

def correlation_graph(X_tr, corr_threshold):
    corr = np.nan_to_num(np.corrcoef(X_tr, rowvar=False))
    n_feat = corr.shape[0]
    G = nx.Graph(); G.add_nodes_from(range(n_feat))
    for i in range(n_feat):
        for j in range(i + 1, n_feat):
            if abs(corr[i, j]) >= corr_threshold:
                G.add_edge(i, j)
    return G

def build_feature_complex(X_tr, corr_threshold=CORR_THRESHOLD, with_B2=False):
    """Nodes = features, edges = |Pearson r| >= threshold, computed on training data only."""
    return _complex_from_graph(correlation_graph(X_tr, corr_threshold), X_tr.shape[1], with_B2)

def build_control_complex(X_tr, kind, corr_threshold=CORR_THRESHOLD):
    """'random': same number of edges as the correlation graph, placed at random; 'empty': no edges."""
    n_feat = X_tr.shape[1]
    if kind == "random":
        n_edges = correlation_graph(X_tr, corr_threshold).number_of_edges()
        G = nx.gnm_random_graph(n_feat, n_edges, seed=int(np.random.randint(2**31 - 1)))
    else:
        G = nx.empty_graph(n_feat)
    return _complex_from_graph(G, n_feat)

def normalized_L0(cplx):
    L = cplx["B1"] @ cplx["B1"].T
    d = np.diag(L).copy(); d[d == 0] = 1.0
    D = np.diag(1.0 / np.sqrt(d))
    return D @ L @ D

def normalized_L0_sparse(Xd, k=K_NEIGHBOURS):
    """Same operator as normalized_L0(build_patient_complex(Xd, k)), built sparsely for large patient graphs."""
    knn = kneighbors_graph(Xd, n_neighbors=k, mode="connectivity", include_self=False)
    A = ((knn + knn.T) > 0).astype(np.float32)
    deg = np.asarray(A.sum(axis=1)).ravel()
    inv_sqrt = 1.0 / np.sqrt(np.where(deg > 0, deg, 1.0))
    Dm = sp.diags(inv_sqrt.astype(np.float32))
    L = (sp.diags((deg > 0).astype(np.float32)) - Dm @ A @ Dm).tocoo()
    idx = torch.tensor(np.vstack([L.row, L.col]), dtype=torch.long)
    return torch.sparse_coo_tensor(idx, torch.tensor(L.data, dtype=torch.float32), L.shape).coalesce()

def _scaled(M):
    if M.size == 0: return M.astype(np.float32)
    lam = np.linalg.eigvalsh(M).max()
    return (M / lam if lam > 1e-8 else M).astype(np.float32)

def edge_laplacians(cplx):
    """Lower (B1ᵀB1) and upper (B2B2ᵀ) parts of the 1st-order Hodge Laplacian, each scaled to spectral radius 1."""
    B1, B2 = cplx["B1"], cplx["B2"]
    return _scaled(B1.T @ B1), _scaled(B2 @ B2.T)

def gcn_adjacency(cplx):
    A = np.eye(cplx["n_nodes"], dtype=np.float32)
    for u, v in cplx["edges"]:
        A[u, v] = A[v, u] = 1.0
    Dm = np.diag(1.0 / np.sqrt(A.sum(axis=1)))
    return (Dm @ A @ Dm).astype(np.float32)

def full_metrics(y_true, preds, probs):
    tn, fp, fn, tp = confusion_matrix(y_true, preds, labels=[0, 1]).ravel()
    return {"Accuracy": accuracy_score(y_true, preds),
            "Sensitivity": tp / (tp + fn) if (tp + fn) else np.nan,
            "Specificity": tn / (tn + fp) if (tn + fp) else np.nan,
            "F1": f1_score(y_true, preds, zero_division=0),
            "AUC": roc_auc_score(y_true, probs) if len(np.unique(y_true)) == 2 else np.nan}

def complex_stats(Xd):
    Xs = StandardScaler().fit_transform(Xd)
    rows = []
    for th in sorted(set(THRESHOLDS + [CORR_THRESHOLD])):
        c = build_feature_complex(Xs, th)
        rows.append({"threshold": th, "nodes": c["n_nodes"], "edges": c["n_edges"], "triangles": c["n_tri"]})
    return pd.DataFrame(rows)

print("Cleveland feature complex by threshold:"); print(complex_stats(X).to_string(index=False))
if RUN_LARGE:
    print("\nCardio feature complex by threshold:"); print(complex_stats(X_cardio).to_string(index=False))

## 4. Models

The original ORB-BSCNN classes are unchanged, except that the patient-graph convolution also accepts a sparse Laplacian. New: the edge-level simplicial convolution, the node + edge simplicial model, and batched GCN and GAT baselines.

In [ ]:
class BinarizeSTE(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)
        return torch.sign(x + 1e-12)
    @staticmethod
    def backward(ctx, grad_output):
        (x,) = ctx.saved_tensors
        grad = grad_output.clone()
        grad[x.abs() > 1] = 0
        return grad
binarize = BinarizeSTE.apply


class BiSimplicialConv(nn.Module):
    """Transductive version: X is [n_nodes, dim] over one fixed patient graph (dense or sparse Laplacian)."""
    def __init__(self, in_dim, out_dim, laplacian, k_hops=2):
        super().__init__()
        self.register_buffer("L", laplacian)
        self.k_hops = k_hops
        self.theta = nn.Parameter(torch.randn(k_hops + 1, in_dim, out_dim) * 0.1)
        self.bn = nn.BatchNorm1d(out_dim)
    def forward(self, X):
        out = 0; Lx = X
        for h in range(self.k_hops + 1):
            out = out + Lx @ self.theta[h]
            Lx = torch.sparse.mm(self.L, Lx) if self.L.is_sparse else self.L @ Lx
        return self.bn(out)


class BatchedBiSimplicialConv(nn.Module):
    """Batched version: X is [batch, n_nodes, dim] over one fixed feature graph."""
    def __init__(self, in_dim, out_dim, laplacian, k_hops=2):
        super().__init__()
        self.register_buffer("L", laplacian)
        self.k_hops = k_hops
        self.theta = nn.Parameter(torch.randn(k_hops + 1, in_dim, out_dim) * 0.1)
        self.bn = nn.BatchNorm1d(out_dim)
    def forward(self, X):
        out = 0; Lx = X
        for h in range(self.k_hops + 1):
            out = out + Lx @ self.theta[h]
            Lx = torch.einsum("ij,bjf->bif", self.L, Lx)
        b, n, f = out.shape
        return self.bn(out.reshape(b * n, f)).reshape(b, n, f)


class BatchedEdgeSimplicialConv(nn.Module):
    """Edge signals [batch, n_edges, dim] filtered with the lower and upper Hodge Laplacians (Yang et al., 2022):
    out = X Θ0 + sum_h L_down^h X Θd_h + sum_h L_up^h X Θu_h.  use_upper=False drops the triangle (L_up) term."""
    def __init__(self, in_dim, out_dim, L_down, L_up, k_hops=2, use_upper=True):
        super().__init__()
        self.register_buffer("L_down", L_down)
        self.register_buffer("L_up", L_up)
        self.k_hops, self.use_upper = k_hops, use_upper
        self.theta0 = nn.Parameter(torch.randn(in_dim, out_dim) * 0.1)
        self.theta_d = nn.Parameter(torch.randn(k_hops, in_dim, out_dim) * 0.1)
        self.theta_u = nn.Parameter(torch.randn(k_hops, in_dim, out_dim) * 0.1) if use_upper else None
        self.bn = nn.BatchNorm1d(out_dim)
    def forward(self, X):
        out = X @ self.theta0; Xd = X; Xu = X
        for h in range(self.k_hops):
            Xd = torch.einsum("ij,bjf->bif", self.L_down, Xd); out = out + Xd @ self.theta_d[h]
            if self.use_upper:
                Xu = torch.einsum("ij,bjf->bif", self.L_up, Xu); out = out + Xu @ self.theta_u[h]
        b, n, f = out.shape
        return self.bn(out.reshape(b * n, f)).reshape(b, n, f)


class ResidualBlock(nn.Module):
    def __init__(self, conv, use_binarize=True, use_residual=True):
        super().__init__()
        self.conv = conv
        self.act = nn.PReLU()
        self.use_binarize = use_binarize
        self.use_residual = use_residual
    def forward(self, X):
        pre = self.conv(X)
        h = self.act(binarize(pre) + pre) if self.use_binarize else self.act(pre)
        return h + X if self.use_residual else h


def _head(hidden_dim, n_classes=2):
    return nn.Sequential(nn.Linear(hidden_dim, hidden_dim // 2), nn.PReLU(), nn.Dropout(0.3),
                         nn.Linear(hidden_dim // 2, n_classes))

class ORB_BSCNN(nn.Module):
    """Patient-graph model (transductive)."""
    def __init__(self, in_dim, hidden_dim, laplacian, n_blocks=3, k_hops=2, use_binarize=True, use_residual=True):
        super().__init__()
        self.input_proj = nn.Linear(in_dim, hidden_dim)
        self.blocks = nn.ModuleList([ResidualBlock(BiSimplicialConv(hidden_dim, hidden_dim, laplacian, k_hops),
                                                   use_binarize, use_residual) for _ in range(n_blocks)])
        self.classifier = _head(hidden_dim)
    def forward(self, X):
        h = self.input_proj(X)
        for block in self.blocks:
            h = block(h)
        return self.classifier(h)

class FeatureGraphORB_BSCNN(nn.Module):
    """Feature-graph model (inductive): each patient is a signal over the feature graph."""
    def __init__(self, hidden_dim, laplacian, n_blocks=2, k_hops=2, use_binarize=True, use_residual=True):
        super().__init__()
        self.input_proj = nn.Linear(1, hidden_dim)
        self.blocks = nn.ModuleList([ResidualBlock(BatchedBiSimplicialConv(hidden_dim, hidden_dim, laplacian, k_hops),
                                                   use_binarize, use_residual) for _ in range(n_blocks)])
        self.classifier = _head(hidden_dim)
    def forward(self, X):
        h = self.input_proj(X.unsqueeze(-1))
        for block in self.blocks:
            h = block(h)
        return self.classifier(h.mean(dim=1))

class SimplicialORB_BSCNN(nn.Module):
    """Node branch over L0 (as in the feature-graph model) plus an edge branch over L1 = L_down + L_up.
    Edge inputs are lifted from the patient's node signal: for edge (u, v) the channels are x_u * x_v and x_v - x_u.
    The two branch readouts (means over nodes and over edges) are concatenated before the classification head."""
    def __init__(self, hidden_dim, L0, L_down, L_up, edges, n_blocks=2, k_hops=2, use_upper=True,
                 use_binarize=True, use_residual=True):
        super().__init__()
        self.node_proj = nn.Linear(1, hidden_dim)
        self.node_blocks = nn.ModuleList([ResidualBlock(BatchedBiSimplicialConv(hidden_dim, hidden_dim, L0, k_hops),
                                                        use_binarize, use_residual) for _ in range(n_blocks)])
        self.has_edges = len(edges) > 0
        if self.has_edges:
            self.register_buffer("src", torch.tensor([u for u, _ in edges], dtype=torch.long))
            self.register_buffer("dst", torch.tensor([v for _, v in edges], dtype=torch.long))
            self.edge_proj = nn.Linear(2, hidden_dim)
            self.edge_blocks = nn.ModuleList([ResidualBlock(BatchedEdgeSimplicialConv(hidden_dim, hidden_dim, L_down, L_up,
                                                                                      k_hops, use_upper),
                                                            use_binarize, use_residual) for _ in range(n_blocks)])
        self.classifier = _head(2 * hidden_dim)
    def forward(self, X):
        h = self.node_proj(X.unsqueeze(-1))
        for block in self.node_blocks:
            h = block(h)
        node_read = h.mean(dim=1)
        if self.has_edges:
            xs, xd = X[:, self.src], X[:, self.dst]
            e = self.edge_proj(torch.stack([xs * xd, xd - xs], dim=-1))
            for block in self.edge_blocks:
                e = block(e)
            edge_read = e.mean(dim=1)
        else:
            edge_read = torch.zeros_like(node_read)
        return self.classifier(torch.cat([node_read, edge_read], dim=1))

class BatchedGCN(nn.Module):
    """GCN (Kipf and Welling) over the feature graph, full precision; same readout and head as the ORB-BSCNN."""
    def __init__(self, hidden_dim, A_hat, n_layers=2, dropout=0.3):
        super().__init__()
        self.register_buffer("A", A_hat)
        self.input_proj = nn.Linear(1, hidden_dim)
        self.layers = nn.ModuleList([nn.Linear(hidden_dim, hidden_dim) for _ in range(n_layers)])
        self.drop = nn.Dropout(dropout)
        self.classifier = _head(hidden_dim)
    def forward(self, X):
        h = self.input_proj(X.unsqueeze(-1))
        for lin in self.layers:
            h = self.drop(F.relu(torch.einsum("ij,bjf->bif", self.A, lin(h))))
        return self.classifier(h.mean(dim=1))

class BatchedGATLayer(nn.Module):
    """Multi-head graph attention (Veličković et al.) restricted to graph neighbours plus self-loops."""
    def __init__(self, in_dim, out_dim, mask, heads=4):
        super().__init__()
        assert out_dim % heads == 0
        self.heads, self.d = heads, out_dim // heads
        self.W = nn.Linear(in_dim, out_dim, bias=False)
        self.a_src = nn.Parameter(torch.randn(heads, self.d) * 0.1)
        self.a_dst = nn.Parameter(torch.randn(heads, self.d) * 0.1)
        self.register_buffer("mask", mask)
    def forward(self, X):
        b, n, _ = X.shape
        Wh = self.W(X).view(b, n, self.heads, self.d)
        s = (Wh * self.a_src).sum(-1); t = (Wh * self.a_dst).sum(-1)              # [b, n, heads]
        e = F.leaky_relu(s.unsqueeze(2) + t.unsqueeze(1), 0.2)                     # [b, i, j, heads]
        e = e.masked_fill(~self.mask[None, :, :, None], float("-inf"))
        att = torch.softmax(e, dim=2)
        return torch.einsum("bijh,bjhd->bihd", att, Wh).reshape(b, n, self.heads * self.d)

class BatchedGAT(nn.Module):
    def __init__(self, hidden_dim, mask, n_layers=2, heads=4, dropout=0.3):
        super().__init__()
        self.input_proj = nn.Linear(1, hidden_dim)
        self.layers = nn.ModuleList([BatchedGATLayer(hidden_dim, hidden_dim, mask, heads) for _ in range(n_layers)])
        self.drop = nn.Dropout(dropout)
        self.classifier = _head(hidden_dim)
    def forward(self, X):
        h = self.input_proj(X.unsqueeze(-1))
        for layer in self.layers:
            h = self.drop(F.elu(layer(h)))
        return self.classifier(h.mean(dim=1))

class MLP(nn.Module):
    def __init__(self, in_dim, hidden=32, n_classes=2):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, hidden), nn.ReLU(),
                                 nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, n_classes))
    def forward(self, x): return self.net(x)

def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

In [ ]:
criterion = nn.CrossEntropyLoss()
_t = lambda a, dt=torch.float32: torch.tensor(a, dtype=dt, device=device)

def _fit(model, forward_train, y_train_t, lr, epochs, adamw=True):
    """Full-batch training. The scheduler monitors the TRAINING loss; no test data are used."""
    if adamw:
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=15)
    else:
        opt = torch.optim.Adam(model.parameters(), lr=lr); sched = None
    model.train()
    for _ in range(epochs):
        opt.zero_grad()
        loss = criterion(forward_train(), y_train_t)
        loss.backward(); opt.step()
        if sched is not None: sched.step(loss.item())
    model.eval()

def _predict(logits, y_te):
    probs = F.softmax(logits, dim=1)[:, 1].cpu().numpy()
    preds = logits.argmax(dim=1).cpu().numpy()
    return full_metrics(y_te, preds, probs)

def train_patient_graph(X_tr, y_tr, X_te, y_te, k=K_NEIGHBOURS, hidden_dim=32, n_blocks=3, k_hops=2,
                        lr=1e-2, epochs=EPOCHS, use_binarize=True, use_residual=True):
    """Transductive: the graph contains training and test patients; only training labels are used."""
    X_a = np.vstack([X_tr, X_te]); n_tr = X_tr.shape[0]
    if X_a.shape[0] >= SPARSE_MIN_NODES:
        L = normalized_L0_sparse(X_a, k=k).to(device)
    else:
        L = _t(normalized_L0(build_patient_complex(X_a, k=k)))
    Xat = _t(X_a); ytr = _t(y_tr, torch.long)
    m = ORB_BSCNN(X_a.shape[1], hidden_dim, L, n_blocks, k_hops, use_binarize, use_residual).to(device)
    _fit(m, lambda: m(Xat)[:n_tr], ytr, lr, epochs)
    with torch.no_grad():
        return _predict(m(Xat)[n_tr:], y_te)

def train_batched(m, X_tr, y_tr, X_te, y_te, lr=1e-2, epochs=EPOCHS):
    """Inductive models over a fixed feature graph: each patient is one graph signal."""
    m = m.to(device)
    Xtr = _t(X_tr); ytr = _t(y_tr, torch.long); Xte = _t(X_te)
    _fit(m, lambda: m(Xtr), ytr, lr, epochs)
    with torch.no_grad():
        return _predict(m(Xte), y_te)

def train_feature_graph(X_tr, y_tr, X_te, y_te, corr_threshold=CORR_THRESHOLD, control=None, hidden_dim=32,
                        n_blocks=2, k_hops=2, use_binarize=True, use_residual=True):
    cplx = build_control_complex(X_tr, control, corr_threshold) if control else build_feature_complex(X_tr, corr_threshold)
    m = FeatureGraphORB_BSCNN(hidden_dim, _t(normalized_L0(cplx)), n_blocks, k_hops, use_binarize, use_residual)
    return train_batched(m, X_tr, y_tr, X_te, y_te)

def train_simplicial(X_tr, y_tr, X_te, y_te, corr_threshold=CORR_THRESHOLD, use_upper=True, hidden_dim=32):
    cplx = build_feature_complex(X_tr, corr_threshold, with_B2=True)
    Ld, Lu = edge_laplacians(cplx)
    m = SimplicialORB_BSCNN(hidden_dim, _t(normalized_L0(cplx)), _t(Ld), _t(Lu), cplx["edges"], use_upper=use_upper)
    return train_batched(m, X_tr, y_tr, X_te, y_te)

def train_gcn(X_tr, y_tr, X_te, y_te, corr_threshold=CORR_THRESHOLD, hidden_dim=32):
    cplx = build_feature_complex(X_tr, corr_threshold)
    return train_batched(BatchedGCN(hidden_dim, _t(gcn_adjacency(cplx))), X_tr, y_tr, X_te, y_te)

def train_gat(X_tr, y_tr, X_te, y_te, corr_threshold=CORR_THRESHOLD, hidden_dim=32):
    cplx = build_feature_complex(X_tr, corr_threshold)
    mask = torch.tensor(gcn_adjacency(cplx) > 0, device=device)
    return train_batched(BatchedGAT(hidden_dim, mask), X_tr, y_tr, X_te, y_te)

def train_feature_graph_nested(X_tr, y_tr, X_te, y_te):
    """Choose τ by inner stratified CV on the outer training fold only, then refit on the whole training fold."""
    inner_seed = int(np.random.randint(2**31 - 1))      # derived from the run seed set before this call
    skf = StratifiedKFold(n_splits=NESTED_INNER, shuffle=True, random_state=inner_seed)
    scores = {}
    for th in THRESHOLDS:
        aucs = []
        for i, (itr, iva) in enumerate(skf.split(X_tr, y_tr)):
            set_seed(inner_seed + i)
            aucs.append(train_feature_graph(X_tr[itr], y_tr[itr], X_tr[iva], y_tr[iva], corr_threshold=th)["AUC"])
        scores[th] = np.mean(aucs)
    best = max(scores, key=scores.get)
    set_seed(inner_seed)
    return {**train_feature_graph(X_tr, y_tr, X_te, y_te, corr_threshold=best), "selected_tau": best}

def train_mlp(X_tr, y_tr, X_te, y_te, epochs=EPOCHS):
    Xtr = _t(X_tr); ytr = _t(y_tr, torch.long); Xte = _t(X_te)
    m = MLP(X_tr.shape[1]).to(device)
    _fit(m, lambda: m(Xtr), ytr, 1e-3, epochs, adamw=False)
    with torch.no_grad():
        return _predict(m(Xte), y_te)

def train_rf(X_tr, y_tr, X_te, y_te, seed=BASE_SEED):
    rf = RandomForestClassifier(n_estimators=300, random_state=seed, n_jobs=-1).fit(X_tr, y_tr)
    return full_metrics(y_te, rf.predict(X_te), rf.predict_proba(X_te)[:, 1])

def train_logreg(X_tr, y_tr, X_te, y_te):
    lr_ = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
    return full_metrics(y_te, lr_.predict(X_te), lr_.predict_proba(X_te)[:, 1])

def train_xgb(X_tr, y_tr, X_te, y_te, seed=BASE_SEED):
    """Fixed, untuned settings, chosen before running (as for the other baselines)."""
    xg = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                       tree_method="hist", random_state=seed, n_jobs=-1, eval_metric="logloss").fit(X_tr, y_tr)
    return full_metrics(y_te, xg.predict(X_te), xg.predict_proba(X_te)[:, 1])

MODEL_FUNCS = {
    PG:                    lambda a, b, c, d: train_patient_graph(a, b, c, d),
    FG:                    lambda a, b, c, d: train_feature_graph(a, b, c, d),
    "MLP":                 lambda a, b, c, d: train_mlp(a, b, c, d),
    "Random Forest":       lambda a, b, c, d: train_rf(a, b, c, d),
    "Logistic Regression": lambda a, b, c, d: train_logreg(a, b, c, d),
    "Ablation: feature graph, no binarization": lambda a, b, c, d: train_feature_graph(a, b, c, d, use_binarize=False),
    "Ablation: feature graph, no residual":     lambda a, b, c, d: train_feature_graph(a, b, c, d, use_residual=False),
    "Ablation: patient graph, 2 blocks":        lambda a, b, c, d: train_patient_graph(a, b, c, d, n_blocks=2),
    SC:                                               lambda a, b, c, d: train_simplicial(a, b, c, d),
    "Ablation: simplicial, no triangles (no L1 up)":  lambda a, b, c, d: train_simplicial(a, b, c, d, use_upper=False),
    "Control: feature model, random graph":           lambda a, b, c, d: train_feature_graph(a, b, c, d, control="random"),
    "Control: feature model, empty graph":            lambda a, b, c, d: train_feature_graph(a, b, c, d, control="empty"),
    "GCN (feature graph)":                            lambda a, b, c, d: train_gcn(a, b, c, d),
    "GAT (feature graph)":                            lambda a, b, c, d: train_gat(a, b, c, d),
    "XGBoost":                                        lambda a, b, c, d: train_xgb(a, b, c, d),
    "ORB-BSCNN (feature graph, nested τ)":            lambda a, b, c, d: train_feature_graph_nested(a, b, c, d),
}
ORIGINAL_MODELS = [PG, FG, "MLP", "Random Forest", "Logistic Regression",
                   "Ablation: feature graph, no binarization", "Ablation: feature graph, no residual",
                   "Ablation: patient graph, 2 blocks"]
NEW_MODELS = [SC, "Ablation: simplicial, no triangles (no L1 up)", "GCN (feature graph)", "GAT (feature graph)",
              "Control: feature model, random graph", "Control: feature model, empty graph", "XGBoost"]
NESTED = ["ORB-BSCNN (feature graph, nested τ)"]
REPORT_ORDER = [SC, "Ablation: simplicial, no triangles (no L1 up)", FG, "ORB-BSCNN (feature graph, nested τ)",
                "Ablation: feature graph, no binarization", "Ablation: feature graph, no residual",
                "Control: feature model, random graph", "Control: feature model, empty graph",
                "GCN (feature graph)", "GAT (feature graph)", PG, "Ablation: patient graph, 2 blocks",
                "MLP", "Random Forest", "XGBoost", "Logistic Regression"]

def prepare_fold(Xtr_raw, Xte_raw):
    """Median imputation and standardization, both fitted on the training fold only."""
    med = np.nanmedian(Xtr_raw, axis=0)
    med = np.where(np.isnan(med), 0.0, med)
    Xtr = np.where(np.isnan(Xtr_raw), med, Xtr_raw)
    Xte = np.where(np.isnan(Xte_raw), med, Xte_raw)
    sc = StandardScaler().fit(Xtr)
    return sc.transform(Xtr).astype(np.float32), sc.transform(Xte).astype(np.float32)

def run_models(Xtr, ytr, Xte, yte, repeat, fold, models):
    out = {}
    for name in models:
        set_seed(run_seed(repeat, fold, name))
        t0 = time.time()
        out[name] = {**MODEL_FUNCS[name](Xtr, ytr, Xte, yte), "fit_seconds": time.time() - t0}
    return out

def repeated_cv(Xd, yd, models, n_repeats, label=""):
    rows = []; t0 = time.time()
    for rep in range(n_repeats):
        skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=BASE_SEED + rep)
        for fold, (tr, te) in enumerate(skf.split(Xd, yd)):
            Xtr, Xte = prepare_fold(Xd[tr], Xd[te])
            res = run_models(Xtr, yd[tr], Xte, yd[te], rep, fold, models)
            for name, m in res.items():
                rows.append({"repeat": rep, "fold": fold, "model": name, **m})
            print(f"{label} repeat {rep + 1}/{n_repeats} fold {fold + 1}/{N_SPLITS} done ({time.time() - t0:.0f}s)")
    return pd.DataFrame(rows)

def summarize(per_fold, order=None):
    g = per_fold.groupby("model")[METRICS].agg(["mean", "std"])
    if order: g = g.loc[[m for m in order if m in g.index]]
    g.columns = [f"{m}_{s}" for m, s in g.columns]
    return g.round(4)

## 5. Significance tests

Same tests as the original notebook, on matched folds: the corrected resampled t-test (Nadeau and Bengio) is the one to report. Each comparison below answers one question; the first column of the table names it.

In [ ]:
def paired_tests(per_fold, model_a, model_b, metric, test_train_ratio=1.0 / (N_SPLITS - 1)):
    wide = per_fold.pivot_table(index=["repeat", "fold"], columns="model", values=metric)
    a, b = wide[model_a].values, wide[model_b].values
    d = a - b; J = len(d); var = d.var(ddof=1) if J > 1 else 0.0
    if var > 0:
        t_corr = d.mean() / np.sqrt((1.0 / J + test_train_ratio) * var)
        p_corr = 2 * stats.t.sf(abs(t_corr), J - 1)
        t_naive, p_naive = stats.ttest_rel(a, b)
    else:
        t_corr = p_corr = t_naive = p_naive = np.nan
    try:
        _, p_w = stats.wilcoxon(a, b)
    except ValueError:
        p_w = np.nan
    return {"Comparison": f"{model_a} vs. {model_b}", "Metric": metric, "Mean A": a.mean(), "Mean B": b.mean(),
            "Mean diff": d.mean(), "Folds A>B": int((d > 0).sum()), "Folds tied": int((d == 0).sum()), "Folds": J,
            "Corrected t": t_corr, "Corrected p": p_corr, "Paired t": t_naive, "Paired t p": p_naive, "Wilcoxon p": p_w}

QUESTIONS = [
    ("Do triangles (2-simplices) add anything?",       SC, "Ablation: simplicial, no triangles (no L1 up)"),
    ("Does edge-level propagation add anything?",      SC, FG),
    ("Does the correlation structure matter? (random)", FG, "Control: feature model, random graph"),
    ("Does any graph matter? (empty)",                 FG, "Control: feature model, empty graph"),
    ("ORB-BSCNN vs. standard GNN (GCN)",               FG, "GCN (feature graph)"),
    ("ORB-BSCNN vs. standard GNN (GAT)",               FG, "GAT (feature graph)"),
    ("Feature vs. patient complex",                    FG, PG),
    ("Binarization",                                   FG, "Ablation: feature graph, no binarization"),
    ("Residual connection",                            FG, "Ablation: feature graph, no residual"),
    ("Fixed vs. nested τ",                             FG, "ORB-BSCNN (feature graph, nested τ)"),
    ("Simplicial vs. logistic regression",             SC, "Logistic Regression"),
    ("Simplicial vs. XGBoost",                         SC, "XGBoost"),
    ("Simplicial vs. Random Forest",                   SC, "Random Forest"),
    ("Simplicial vs. MLP",                             SC, "MLP"),
    ("Feature graph vs. logistic regression",          FG, "Logistic Regression"),
    ("Feature graph vs. XGBoost",                      FG, "XGBoost"),
]

def significance_table(per_fold, metric="AUC"):
    present = set(per_fold["model"])
    rows = [{"Question": q, **paired_tests(per_fold, a, b, metric)} for q, a, b in QUESTIONS if a in present and b in present]
    return pd.DataFrame(rows).round(4)

## 6. Cleveland (n = 297): repeated 5-fold CV, 25 paired folds

The original eight models use the same folds and seeds as the manuscript, so their rows should match Table 2 and the ablation table.

In [ ]:
if RUN_CLEVELAND:
    models = ORIGINAL_MODELS + NEW_MODELS + (NESTED if RUN_NESTED else [])
    cv_cleveland = repeated_cv(X, y, models, N_REPEATS, label="Cleveland")
    table_cleveland = summarize(cv_cleveland, REPORT_ORDER)
    sig_cleveland = significance_table(cv_cleveland)
    if RUN_NESTED:
        tau_cleveland = cv_cleveland.loc[cv_cleveland["model"] == NESTED[0], "selected_tau"].value_counts().sort_index()
        print("τ selected by nested CV (Cleveland):"); print(tau_cleveland.to_string())
    display(table_cleveland)

In [ ]:
if RUN_CLEVELAND:
    display(sig_cleveland)

## 7. Cardiovascular Disease dataset (n ≈ 68,000): repeated 5-fold CV

All models, including the patient graph (sparse kNN Laplacian over all patients, transductive as before).

**Use a GPU runtime for this section** (Runtime → Change runtime type → T4 GPU). On CPU one full-batch fit over ~54,000 training patients takes several minutes, so the whole section would take well over 10 hours; on a T4 it takes roughly 1–2 hours. GPU results can differ from CPU results in the last decimal places; the Cleveland section stays on CPU.

In [ ]:
if RUN_LARGE:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Large-dataset device:", device)
    if device.type == "cpu" and not CARDIO_SUBSAMPLE:
        print("WARNING: no GPU found. Switch to a GPU runtime, or set CARDIO_SUBSAMPLE (e.g. 10000) and re-run section 2.")
    models = ORIGINAL_MODELS + NEW_MODELS + (NESTED if RUN_NESTED else [])
    cv_cardio = repeated_cv(X_cardio, y_cardio, models, LARGE_REPEATS, label="Cardio")
    table_cardio = summarize(cv_cardio, REPORT_ORDER)
    sig_cardio = significance_table(cv_cardio)
    if RUN_NESTED:
        tau_cardio = cv_cardio.loc[cv_cardio["model"] == NESTED[0], "selected_tau"].value_counts().sort_index()
        print("τ selected by nested CV (Cardio):"); print(tau_cardio.to_string())
    display(table_cardio)

In [ ]:
if RUN_LARGE:
    display(sig_cardio)

## 8. Model size and training time

Fit times for the large dataset were measured on the GPU (if one was used) and are not comparable with the Cleveland CPU times.

In [ ]:
device = torch.device("cpu")

def size_table():
    Xtr, _ = prepare_fold(X, X)
    c = build_feature_complex(Xtr, CORR_THRESHOLD, with_B2=True); Ld, Lu = edge_laplacians(c)
    L0 = _t(normalized_L0(c))
    models = {FG: FeatureGraphORB_BSCNN(32, L0),
              SC: SimplicialORB_BSCNN(32, L0, _t(Ld), _t(Lu), c["edges"]),
              "Ablation: simplicial, no triangles (no L1 up)": SimplicialORB_BSCNN(32, L0, _t(Ld), _t(Lu), c["edges"], use_upper=False),
              "GCN (feature graph)": BatchedGCN(32, _t(gcn_adjacency(c))),
              "GAT (feature graph)": BatchedGAT(32, torch.tensor(gcn_adjacency(c) > 0)),
              "MLP": MLP(X.shape[1])}
    rows = [{"Model": k, "Trainable parameters (Cleveland, τ = 0.15)": count_params(m)} for k, m in models.items()]
    return pd.DataFrame(rows)

table_size = size_table()
timing = pd.concat([d.assign(dataset=n) for n, d in
                    [("Cleveland", cv_cleveland if RUN_CLEVELAND else None), ("Cardio", cv_cardio if RUN_LARGE else None)]
                    if d is not None])
table_time = timing.groupby(["dataset", "model"])["fit_seconds"].mean().unstack(0).round(2)
table_size

In [ ]:
table_time

## 9. Save all results

In [ ]:
out = "ORB_BSCNN_optionB_results.xlsx"
settings = pd.DataFrame({"setting": ["QUICK_TEST", "BASE_SEED", "N_REPEATS", "LARGE_REPEATS", "N_SPLITS", "NESTED_INNER",
                                     "EPOCHS", "CORR_THRESHOLD", "K_NEIGHBOURS", "THRESHOLDS", "CARDIO_SUBSAMPLE"],
                         "value": [QUICK_TEST, BASE_SEED, N_REPEATS, LARGE_REPEATS, N_SPLITS, NESTED_INNER,
                                   EPOCHS, CORR_THRESHOLD, K_NEIGHBOURS, str(THRESHOLDS), str(CARDIO_SUBSAMPLE)]})
with pd.ExcelWriter(out) as xw:
    settings.to_excel(xw, sheet_name="Settings", index=False)
    versions.to_excel(xw, sheet_name="Versions", index=False)
    complex_stats(X).to_excel(xw, sheet_name="Complex_Cleveland", index=False)
    if RUN_CLEVELAND:
        table_cleveland.to_excel(xw, sheet_name="Cleveland_CV")
        sig_cleveland.to_excel(xw, sheet_name="Cleveland_significance", index=False)
        cv_cleveland.to_excel(xw, sheet_name="PerFold_Cleveland", index=False)
    if RUN_LARGE:
        complex_stats(X_cardio).to_excel(xw, sheet_name="Complex_Cardio", index=False)
        table_cardio.to_excel(xw, sheet_name="Cardio_CV")
        sig_cardio.to_excel(xw, sheet_name="Cardio_significance", index=False)
        cv_cardio.to_excel(xw, sheet_name="PerFold_Cardio", index=False)
    table_size.to_excel(xw, sheet_name="Model_size", index=False)
    table_time.to_excel(xw, sheet_name="Fit_time_seconds")
print("Saved", out)
try:
    from google.colab import files
    files.download(out)
except Exception:
    pass